<a href="https://colab.research.google.com/github/YrD715/Simple_ACE_Step_Colab/blob/main/Simple_ACE_Step_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

In [ ]:
!pip install --upgrade --force-reinstall --no-cache-dir \
  torch torchvision torchaudio \
  --index-url https://download.pytorch.org/whl/cu128

In [ ]:
import torch

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import os
from pathlib import Path

%cd /content

!apt-get update -y
!apt-get install -y git ffmpeg aria2

if not Path("/content/ComfyUI").exists():
    !git clone https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI

%cd /content/ComfyUI
!pip install -r requirements.txt

%cd /content/ComfyUI/custom_nodes

if not Path("/content/ComfyUI/custom_nodes/ACE-Step-ComfyUI").exists():
    !git clone https://github.com/ace-step/ACE-Step-ComfyUI.git

%cd /content/ComfyUI

req = Path("/content/ComfyUI/custom_nodes/ACE-Step-ComfyUI/requirements.txt")
if req.exists():
    !pip install -r /content/ComfyUI/custom_nodes/ACE-Step-ComfyUI/requirements.txt

In [ ]:
from pathlib import Path

COMFY = Path("/content/ComfyUI")
MODELS = COMFY / "models"

for folder in ["diffusion_models", "text_encoders", "vae"]:
    (MODELS / folder).mkdir(parents=True, exist_ok=True)

downloads = [
    (
        "https://huggingface.co/Comfy-Org/ace_step_1.5_ComfyUI_files/resolve/main/split_files/diffusion_models/acestep_v1.5_xl_turbo_bf16.safetensors",
        MODELS / "diffusion_models" / "acestep_v1.5_xl_turbo_bf16.safetensors",
    ),
    (
        "https://huggingface.co/Comfy-Org/ace_step_1.5_ComfyUI_files/resolve/main/split_files/text_encoders/qwen_0.6b_ace15.safetensors",
        MODELS / "text_encoders" / "qwen_0.6b_ace15.safetensors",
    ),
    (
        "https://huggingface.co/Comfy-Org/ace_step_1.5_ComfyUI_files/resolve/main/split_files/text_encoders/qwen_4b_ace15.safetensors",
        MODELS / "text_encoders" / "qwen_4b_ace15.safetensors",
    ),
    (
        "https://huggingface.co/Comfy-Org/ace_step_1.5_ComfyUI_files/resolve/main/split_files/vae/ace_1.5_vae.safetensors",
        MODELS / "vae" / "ace_1.5_vae.safetensors",
    ),
]

for url, out_path in downloads:
    if out_path.exists() and out_path.stat().st_size > 1024 * 1024:
        print(f"Already exists: {out_path}")
    else:
        print(f"Downloading: {out_path.name}")
        !aria2c -x 8 -s 8 -k 1M -c "{url}" -d "{out_path.parent}" -o "{out_path.name}"

print("\nVerification:")
for _, out_path in downloads:
    if out_path.exists():
        print(f"OK: {out_path} — {out_path.stat().st_size / 1024**3:.2f} GB")
    else:
        print(f"MISSING: {out_path}")

In [ ]:
%cd /content/ComfyUI

!pkill -f "python main.py" || true

!nohup python main.py --listen 127.0.0.1 --port 8188 > /content/comfyui.log 2>&1 &

import time, requests

for i in range(30):
    try:
        r = requests.get("http://127.0.0.1:8188", timeout=5)
        print("ComfyUI status:", r.status_code)
        break
    except Exception as e:
        print("Waiting for ComfyUI...", i + 1)
        time.sleep(2)
else:
    print("ComfyUI did not start. Showing log:")
    print(open("/content/comfyui.log", "r", errors="ignore").read()[-4000:])

In [ ]:
%%writefile /content/simple_song_app.py
import copy
import glob
import json
import os
import random
import time
import uuid
from pathlib import Path

import gradio as gr
import requests


COMFY_URL = "http://127.0.0.1:8188"
COMFY_DIR = Path("/content/ComfyUI")
OUTPUT_DIR = COMFY_DIR / "output"


def build_prompt(tags, lyrics, duration, seed, bpm, language, keyscale):
    """
    Manual API-format version of the user's ACE-Step 1.5 Turbo workflow.
    This avoids needing the ComfyUI visual editor or API export.
    """

    duration = float(duration)
    seed = int(seed)
    bpm = int(bpm)

    filename_prefix = f"audio/ace_gradio_{uuid.uuid4().hex[:10]}"

    prompt = {
        "104": {
            "class_type": "UNETLoader",
            "inputs": {
                "unet_name": "acestep_v1.5_xl_turbo_bf16.safetensors",
                "weight_dtype": "default",
            },
        },
        "105": {
            "class_type": "DualCLIPLoader",
            "inputs": {
                "clip_name1": "qwen_0.6b_ace15.safetensors",
                "clip_name2": "qwen_4b_ace15.safetensors",
                "type": "ace",
                "device": "default",
            },
        },
        "106": {
            "class_type": "VAELoader",
            "inputs": {
                "vae_name": "ace_1.5_vae.safetensors",
            },
        },
        "78": {
            "class_type": "ModelSamplingAuraFlow",
            "inputs": {
                "model": ["104", 0],
                "shift": 3,
            },
        },
        "98": {
            "class_type": "EmptyAceStep1.5LatentAudio",
            "inputs": {
                "seconds": duration,
                "batch_size": 1,
            },
        },
        "94": {
            "class_type": "TextEncodeAceStepAudio1.5",
            "inputs": {
                "clip": ["105", 0],
                "tags": tags,
                "lyrics": lyrics,
                "seed": seed,
                "bpm": bpm,
                "duration": duration,
                "timesignature": "4",
                "language": language,
                "keyscale": keyscale,
                "generate_audio_codes": True,
                "cfg_scale": 2,
                "temperature": 0.85,
                "top_p": 0.9,
                "top_k": 0,
                "min_p": 0,
            },
        },
        "47": {
            "class_type": "ConditioningZeroOut",
            "inputs": {
                "conditioning": ["94", 0],
            },
        },
        "3": {
            "class_type": "KSampler",
            "inputs": {
                "model": ["78", 0],
                "positive": ["94", 0],
                "negative": ["47", 0],
                "latent_image": ["98", 0],
                "seed": seed,
                "steps": 8,
                "cfg": 1,
                "sampler_name": "euler",
                "scheduler": "simple",
                "denoise": 1,
            },
        },
        "18": {
            "class_type": "VAEDecodeAudio",
            "inputs": {
                "samples": ["3", 0],
                "vae": ["106", 0],
            },
        },
        "107": {
            "class_type": "SaveAudioMP3",
            "inputs": {
                "audio": ["18", 0],
                "filename_prefix": filename_prefix,
                "quality": "V0",
            },
        },
    }

    return prompt, filename_prefix


def find_output_audio(filename_prefix):
    wanted = filename_prefix.split("/")[-1]
    files = glob.glob(str(OUTPUT_DIR / "**" / f"{wanted}*.mp3"), recursive=True)
    files += glob.glob(str(OUTPUT_DIR / "**" / f"{wanted}*.wav"), recursive=True)
    files = [Path(f) for f in files if Path(f).exists()]
    if not files:
        return None
    return str(max(files, key=lambda p: p.stat().st_mtime))


def generate_song(tags, lyrics, duration, seed, bpm, language, keyscale):
    if not tags.strip():
        raise gr.Error("Add a song style / prompt.")

    if not lyrics.strip():
        lyrics = "[verse]\nSing a short test song for me\nMake it simple and catchy"

    if seed in [None, "", 0, "0"]:
        seed = random.randint(1, 2**31 - 1)

    prompt, filename_prefix = build_prompt(
        tags=tags.strip(),
        lyrics=lyrics.strip(),
        duration=duration,
        seed=seed,
        bpm=bpm,
        language=language,
        keyscale=keyscale,
    )

    try:
        r = requests.post(f"{COMFY_URL}/prompt", json={"prompt": prompt}, timeout=30)
        r.raise_for_status()
    except Exception as e:
        log = ""
        try:
            log = open("/content/comfyui.log", "r", errors="ignore").read()[-3000:]
        except Exception:
            pass
        raise gr.Error(f"Could not submit to ComfyUI: {e}\n\nComfyUI log tail:\n{log}")

    data = r.json()
    prompt_id = data.get("prompt_id", "unknown")

    start = time.time()
    timeout_seconds = 30 * 60

    while time.time() - start < timeout_seconds:
        audio_path = find_output_audio(filename_prefix)
        if audio_path:
            return audio_path, f"Done. Seed: {seed}"

        # Check for errors in history if ComfyUI has finished the prompt.
        try:
            h = requests.get(f"{COMFY_URL}/history/{prompt_id}", timeout=10)
            if h.status_code == 200:
                hist = h.json()
                if prompt_id in hist:
                    # If history exists but file is still not visible, give it a moment.
                    audio_path = find_output_audio(filename_prefix)
                    if audio_path:
                        return audio_path, f"Done. Seed: {seed}"
        except Exception:
            pass

        time.sleep(3)

    log = ""
    try:
        log = open("/content/comfyui.log", "r", errors="ignore").read()[-4000:]
    except Exception:
        pass

    raise gr.Error(
        "Timed out waiting for audio. It may have crashed or the T4 may be too slow.\n\n"
        f"Prompt ID: {prompt_id}\n\nComfyUI log tail:\n{log}"
    )


with gr.Blocks(title="Simple ACE-Step Song Maker") as demo:
    gr.Markdown("# Simple ACE-Step 1.5 Turbo Song Maker")
    gr.Markdown("Type the song style, paste lyrics, choose length, and generate. ComfyUI is hidden in the background.")

    with gr.Row():
        with gr.Column(scale=1):
            tags = gr.Textbox(
                label="Song style / prompt",
                lines=4,
                value="Country rock, male vocals, acoustic guitar, drums, bass, funny lyrics, catchy chorus",
            )

            lyrics = gr.Textbox(
                label="Lyrics",
                lines=12,
                value="[Verse 1]\nTesting one two three\nMake a little song for me\n\n[Chorus]\nRun the beat and let it play\nMake some music today",
            )

            duration = gr.Slider(
                label="Duration in seconds",
                minimum=10,
                maximum=600,
                value=30,
                step=5,
            )

            with gr.Row():
                seed = gr.Number(label="Seed, 0 = random", value=0, precision=0)
                bpm = gr.Number(label="BPM", value=95, precision=0)

            with gr.Row():
                language = gr.Dropdown(
                    label="Language",
                    choices=["en", "zh", "ja", "ko", "es", "fr", "de", "it", "pt", "ru"],
                    value="en",
                )
                keyscale = gr.Textbox(label="Key", value="E minor")

            button = gr.Button("Generate Song", variant="primary")

        with gr.Column(scale=1):
            audio = gr.Audio(label="Generated song", type="filepath")
            status = gr.Textbox(label="Status")

    button.click(
        generate_song,
        inputs=[tags, lyrics, duration, seed, bpm, language, keyscale],
        outputs=[audio, status],
    )


if __name__ == "__main__":
    demo.queue(max_size=3).launch(
        server_name="0.0.0.0",
        server_port=7860,
        share=True,
        debug=True,
    )

In [ ]:
!pip install -q gradio

%cd /content

!python simple_song_app.py